# Lezione 26 — Entita' e relazioni: chi e cosa compare nelle memorie

**Come si usa questo notebook.** Come sempre. Prerequisito: Lezione 25
(punteggio composito). Finora ogni memoria e' stata un testo isolato; oggi
ne estraiamo i **nomi propri** (persone, luoghi, nomi di progetti) che vi
compaiono — il primo passo per collegare memorie diverse che parlano
della stessa persona o dello stesso posto, tema della Lezione 27 (il
grafo).

**Cosa saprai fare alla fine:** estrarre entita' da un testo con
un'euristica basata su espressioni regolari, spiegarne onestamente i
limiti rispetto a un vero sistema di NER, e costruire la mappa
entita' -> memorie per l'intero archivio.

## Parte 1 — Teoria: un'euristica dichiarata, non un vero NER

Riconoscere automaticamente "di chi/cosa si parla" in un testo e' un
compito chiamato **Named Entity Recognition (NER)**: sistemi seri (spaCy,
API cloud di NLP) usano modelli addestrati che riconoscono persone, luoghi,
organizzazioni distinguendoli l'uno dall'altro, gestendo entita' composte
da piu' parole e casi ambigui. Questo ambiente non ha ne' una libreria di
NER installata ne' accesso di rete per scaricarne una: costruiamo
un'**euristica dichiarata**, molto piu' semplice e con limiti espliciti,
non un sostituto di un vero NER.

L'euristica: in inglese (la lingua dei testi delle memorie), i nomi
propri iniziano quasi sempre con una lettera maiuscola **e non sono la
prima parola della frase** (perche' anche le frasi normali iniziano con
maiuscola). Un'espressione regolare trova tutte le parole capitalizzate;
poi filtriamo le parole capitalizzate che **non sono nomi propri** ma
iniziano comunque con maiuscola per convenzione grammaticale (qui, `The`
a inizio frase).

**Limiti dichiarati, non nascosti:**

- non distingue persona da luogo da nome di progetto — tutto viene
  trattato come "candidato entita'" generico;
- non gestisce entita' composte da piu' parole (es. "New York" verrebbe
  spezzato in due candidati separati);
- dipende dalla capitalizzazione: su testo tutto minuscolo o in lingue
  senza questa convenzione (l'italiano capitalizza i nomi propri allo
  stesso modo, ma non tutte le lingue lo fanno) non funzionerebbe;
- la lista di parole da escludere (`STOPWORD_MAIUSCOLE`) e' costruita a
  mano su **questo** dataset, non e' una lista universale — su un testo
  diverso andrebbe rivista.

Per lo scopo di questa lezione (collegare memorie che condividono
un'entita', non produrre un'estrazione perfetta) e' sufficiente, a patto
di essere onesti sui suoi limiti.

In [1]:
import re

STOPWORD_MAIUSCOLE = {'The'}  # costruita osservando i falsi positivi su questo dataset


def estrai_entita(testo):
    candidate = re.findall(r'\b[A-Z][a-zA-Z]*\b', str(testo))
    return [c for c in candidate if c not in STOPWORD_MAIUSCOLE]


frase_1 = 'The user visited Glasgow with Marco.'
frase_2 = 'Lucia booked a train to Milano.'
print(frase_1, '->', estrai_entita(frase_1))
print(frase_2, '->', estrai_entita(frase_2))

The user visited Glasgow with Marco. -> ['Glasgow', 'Marco']
Lucia booked a train to Milano. -> ['Lucia', 'Milano']


Leggi l'output: `The`, a inizio frase, e' correttamente escluso (non e' un
nome proprio); `Glasgow` e `Marco` (a meta' frase) sono trovati entrambi
nella prima frase, `Lucia` e `Milano` nella seconda. Nessuna distinzione
tra "persona" e "luogo": e' esattamente il limite dichiarato sopra.

## Parte 2 — Esercizio guidato

Il tuo compito: applica `estrai_entita` a `"Elena met a friend in
Torino."` e verifica che il risultato contenga esattamente due elementi
(`Elena`, `Torino`), senza `The` (che qui non compare nemmeno) e senza
altre parole.

In [2]:
# Scrivi qui: chiama estrai_entita su 'Elena met a friend in Torino.'
# e verifica che il risultato sia ['Elena', 'Torino'].

pass

### Soluzione spiegata

La frase ha due sole parole capitalizzate che non sono la prima della
frase: `Elena` (soggetto) e `Torino` (il luogo, dopo "in"). Nessuna delle
due e' nella lista di esclusione, quindi entrambe sopravvivono al
filtro.

In [3]:
risultato = estrai_entita('Elena met a friend in Torino.')
print(risultato)
assert risultato == ['Elena', 'Torino']

['Elena', 'Torino']


## Parte 3 — Il progetto: Memory AI Lab, passo 26 — la mappa entita' -> memorie

Applichiamo `estrai_entita` a tutto il train set e costruiamo due
strutture che la Lezione 27 (grafo) usera' direttamente: quali memorie
menzionano ciascuna entita', e quali coppie di entita' compaiono **nella
stessa memoria** (una relazione, non ancora un grafo vero e proprio — solo
i dati grezzi da cui costruirlo).

In [4]:
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations

processed = Path('..') / 'datasets' / 'processed'
train = pd.read_csv(processed / 'memory_train.csv')
train['entita'] = train['text'].apply(estrai_entita)

con_entita = (train['entita'].str.len() > 0).sum()
print(f'memorie con almeno un\'entita\': {con_entita} / {len(train)}')

frequenza_entita = Counter(e for lista in train['entita'] for e in lista)
print('\nentita\' piu\' frequenti:')
for entita, conteggio in frequenza_entita.most_common(5):
    print(f'  {entita}: {conteggio} memorie')

memorie con almeno un'entita': 157 / 213

entita' piu' frequenti:
  Lucia: 28 memorie
  Giorgio: 23 memorie
  Glasgow: 22 memorie
  Sara: 22 memorie
  Elena: 19 memorie


In [5]:
mappa_entita_memorie = defaultdict(list)
for _, riga in train.iterrows():
    for entita in riga['entita']:
        mappa_entita_memorie[entita].append(riga['memory_id'])

coppie_co_occorrenza = Counter()
for lista in train['entita']:
    for a, b in combinations(sorted(set(lista)), 2):
        coppie_co_occorrenza[(a, b)] += 1

print(f"entita' distinte trovate: {len(mappa_entita_memorie)}")
print(f'memorie che menzionano "Lucia": {len(mappa_entita_memorie["Lucia"])}')
print('\ncoppie di entita\' co-occorrenti piu\' frequenti:')
for (a, b), conteggio in coppie_co_occorrenza.most_common(5):
    print(f'  ({a}, {b}): {conteggio} memorie in comune')

entita' distinte trovate: 14
memorie che menzionano "Lucia": 28

coppie di entita' co-occorrenti piu' frequenti:
  (Napoli, Sara): 5 memorie in comune
  (Giorgio, Glasgow): 4 memorie in comune
  (Giorgio, TensorFlow): 4 memorie in comune
  (Marco, Napoli): 4 memorie in comune
  (Marco, Milano): 3 memorie in comune


Guarda i numeri: circa il 74% delle memorie (157 su 213 nell'esecuzione
di riferimento) contiene almeno un'entita' riconosciuta dall'euristica —
il resto sono frasi generiche senza nomi propri (es. "The user likes
walking meetings.", nessun nome ne' luogo). Le coppie di co-occorrenza
piu' frequenti mescolano persone e luoghi indistintamente (di nuovo, il
limite dichiarato: l'euristica non sa che una e' una persona e l'altra un
posto) — ma la relazione stessa ("queste due entita' compaiono insieme in
N memorie") e' esattamente l'informazione grezza che la Lezione 27
trasformera' in un grafo navigabile.

## Cosa hai imparato

- Un'**euristica dichiarata** (regex su parole capitalizzate, con
  un'esclusione costruita osservando i falsi positivi) e' un sostituto
  onesto ma limitato di un vero NER quando non ci sono librerie o accesso
  di rete disponibili — dichiarare i limiti e' parte del lavoro, non un
  dettaglio da nascondere.
- La mappa **entita' -> memorie** collega record altrimenti isolati
  attraverso cio' di cui parlano, non solo attraverso la loro similarita'
  testuale (Lezioni 17-18).
- Le **co-occorrenze** (coppie di entita' nella stessa memoria) sono i
  dati grezzi da cui si costruisce un grafo: nodi (entita', memorie) e
  archi (co-occorrenza), tema della prossima lezione.

## Quiz

1. Perche' l'euristica di questa lezione non e' un vero sistema di NER, e
   quali limiti concreti ne derivano?
2. Perche' `STOPWORD_MAIUSCOLE` e' costruita osservando **questo**
   dataset, e cosa rischieresti applicando la stessa lista a un testo
   diverso?
3. Cosa rappresenta una coppia in `coppie_co_occorrenza`, e perche' non e'
   ancora un grafo?

<details>
<summary><b>Apri le risposte</b></summary>

1. Perche' non distingue tipi di entita' (persona/luogo/altro), non
   gestisce entita' multi-parola, e dipende interamente dalla
   capitalizzazione — un vero NER usa modelli addestrati che gestiscono
   tutti questi casi. I limiti concreti: entita' composte spezzate,
   nessuna categorizzazione, fragilita' su testo senza convenzioni di
   capitalizzazione standard.
2. Perche' e' stata costruita guardando i falsi positivi *osservati* su
   questo specifico insieme di testi (qui, solo `The`): un testo diverso
   potrebbe avere altre parole capitalizzate a inizio frase che qui non
   compaiono mai (es. `A`, `His`, `Her`), e la lista andrebbe rivista da
   capo, non riusata ciecamente.
3. Rappresenta due entita' che compaiono **nella stessa memoria** almeno
   una volta, col conteggio di quante volte succede — un dato grezzo di
   co-occorrenza. Non e' ancora un grafo perche' manca la struttura
   esplicita di nodi e archi navigabili (chi e' collegato a chi, con che
   peso): e' il materiale da cui il grafo si costruisce, non il grafo
   stesso.
</details>

## Fonti

- Python, documentazione ufficiale, modulo `re` (espressioni regolari):
  https://docs.python.org/3/library/re.html
- Python, documentazione ufficiale, `itertools.combinations`:
  https://docs.python.org/3/library/itertools.html#itertools.combinations

## Prossima lezione

La mappa entita'-memorie e le co-occorrenze di oggi diventano nodi e archi
di un **grafo** vero e proprio, costruito ed esplorato con `networkx`.